In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

TARGET = 'target_is_fraud'
ID_COL = 'customer_id'

train = pd.read_csv('../6.Data/Yann_Process_train.csv')
test  = pd.read_csv('../6.Data/Yann_Process_test.csv')


def build_features(df):

    df = df.copy()

    df['Score_De_Confiance'] = df['ip_risk_z'] - df['device_trust_z']

    df['Score_Risque'] = (
        df['chargebacks_12m'].clip(lower=0)
        + df['failed_payments_6m'].clip(lower=0)
        + df['ip_risk_z'].clip(lower=0)
        + df['support_tickets_90d'].clip(lower=0)
        - df['device_trust_z'].clip(upper=0)
    )

    df['cb_per_tenure'] = df['chargebacks_12m'] / (df['tenure_months'].abs() + 0.1)

    df['vpn_ip'] = df['is_vpn'] * df['ip_risk_z']

    df['devices_x_vpn'] = df['num_devices_30d'] * df['is_vpn']

    df['nb_chargeback'] = df['chargebacks_12m'] * df['failed_payments_6m']

    return df


train = build_features(train)
test  = build_features(test)


FEATURES = [
    'Score_De_Confiance',
    'ip_risk_z',
    'tenure_months',
    'Score_Risque',
    'is_vpn',
    'cb_per_tenure',
    'vpn_ip',
    'failed_payments_6m',
    'num_devices_30d',
    'devices_x_vpn',
    'support_tickets_90d',
]


X      = train[FEATURES]
y      = train[TARGET]
X_test = test[FEATURES]


X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


# Normalisation
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)


# Modèle SVM
model = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    probability=True,
    class_weight='balanced',
    random_state=42
)

model.fit(X_train, y_train)


THRESHOLD = 0.51

y_prob_val = model.predict_proba(X_val)[:, 1]
y_pred_val = (y_prob_val >= THRESHOLD).astype(int)


repport = classification_report(
    y_val,
    y_pred_val,
    output_dict=True
)


y_test_prob = model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_prob >= THRESHOLD).astype(int)


f1 = f1_score(y_val, y_pred_val)

cm = confusion_matrix(y_val, y_pred_val)

recall = recall_score(y_val, y_pred_val)
print(f"Recall : {recall:.4f}")

precision = precision_score(y_val, y_pred_val)
print(f"Precision : {precision:.4f}")

print(f"F1-score : {f1:.4f}")


submission = pd.DataFrame({
    ID_COL:              test[ID_COL],
    'fraud_probability': y_test_prob,
    TARGET:              y_test_pred,
})


submission.to_csv('../6.Data/submission_svm.csv', index=False)

print(
    f"Submission -> {submission[TARGET].sum()} fraudes | "
    f"{submission[TARGET].mean()*100:.2f}% du test"
)